[![Open In Colab](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/documentation/badge/open-in-colab.svg)](https://colab.research.google.com/github/crunchdao/quickstarters/blob/master/competitions/broad-obesity-3/quickstarters/random/random.ipynb)

![Banner](https://raw.githubusercontent.com/crunchdao/quickstarters/refs/heads/master/competitions/broad-obesity-3/assets/banner.webp)

# Obesity ML Competition: Tackling metabolic diseases

## Crunch 3 – Identifying combinatorial perturbations to drive white and brown adipocyte differentiation

In Crunch 1, we explored the single-cell transcriptomic response to **single-gene** perturbations that were not provided in the training dataset.

In Crunch 2, we explored how well we can predict the single-cell transcriptomic response to **double-gene** perturbations that were not provided in the training dataset.

In Crunch 3, we will nominate combinatorial gene perturbations that are predicted to most strongly increase a target transcriptional program.

# Setup

The first steps to get started are:
1. Get the setup command
2. Execute it in the cell below

### >> https://hub.crunchdao.io/competitions/broad-obesity-3/submit/notebook

![Reveal token](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/documentation/animations/reveal-token.gif)

In [ ]:
%pip install crunch-cli --upgrade --quiet --progress-bar off

In [ ]:
# Necessary for the staging environment
# %env CRUNCH_COMPETITIONS_DIRECTORY_PATH=../../../../
%env CRUNCH_COMPETITIONS_BRANCH=feat/broad-obesity-3
%env API_BASE_URL=https://api.hub.crunchdao.io/
%env WEB_BASE_URL=https://hub.crunchdao.io/

import crunch.store
crunch.store.load_from_env()

In [ ]:
# Install the Crunch CLI
%pip install crunch-cli --upgrade --quiet --progress-bar off

# Setup your local environment
!crunch setup-notebook broad-obesity-3 aaaabbbbccccddddeeeeffff

# Your model

## Setup

In [2]:
import os
from typing import Literal

# Import your dependencies
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

In [ ]:
import crunch

# Load the Crunch Toolings
crunch_tools = crunch.load_notebook()

## Understanding the Data

The data was downloaded when you setup your local environment and is now available in the `data/` directory.

- `predict_perturbations_3.parquet`: The gene pairs to predict.
- `thermogenic_signatures.csv`: A list of genes and their thermogenic signatures.
  
- `TF150_ThermoScores_cell.csv`: The raw and z-scored signature scores for each adipocyte cell for Crunch 1's training set.
- `TF15_MOI2_ThermoScores_cell.csv`: The raw and z-scored signature scores for each adipocyte cell for Crunch 2's training set.

- `TF150_ThermoScores_perturbation.csv`: The perturbation-level score for each perturbation for Crunch 1's training set.
- `TF15_MOI2_ThermoScores_perturbation.csv`: The perturbation-level score for each perturbation for Crunch 2's training set.

### Understanding `predict_perturbations_3.parquet`

Participants will search over all possible two-gene combinations drawn from a set of 2,991 genes, defined as genes with more than 5 transcripts per million together with transcription factors and marker genes.

This yields a total of 4,474,413 possible gene pairs (after filtering out pairs already tested in Crunch 2). 

In [8]:
def _load_predict_perturbations(data_directory_path="data") -> pd.DataFrame:
    return pd.read_parquet(os.path.join(data_directory_path, "predict_perturbations_3.parquet"))


predict_perturbations = _load_predict_perturbations()

print("Left genes count:", predict_perturbations["gene_1"].nunique())
print("Right genes count:", predict_perturbations["gene_2"].nunique())

predict_perturbations

Left genes count: 2991
Right genes count: 2991


,gene_1,gene_2
0,AACS,AACS
1,AACS,AATF
2,AACS,ABCA1
3,AACS,ABCB8
4,AACS,ABCB9
...,...,...
4474408,ZSCAN5A,ZXDC
4474409,ZSCAN5A,ZZEF1
4474410,ZXDC,ZXDC
4474411,ZXDC,ZZEF1


### Understanding `thermogenic_signatures.csv`

For each adipocyte cell and each signature gene set, the EWSC team define 12 signature gene sets relevant to thermogenesis for which they computed an enrichment score using the expression levels of the signature genes relative to the matched control sets.

In [45]:
def _load_thermogenic_signatures(data_directory_path="data") -> pd.DataFrame:
    return pd.read_csv(os.path.join(data_directory_path, "thermogenic_signatures.csv"))


thermogenic_signatures = _load_thermogenic_signatures()
signature_names = thermogenic_signatures["name"].unique()

# Print the number of genes in each signature
thermogenic_signatures.groupby("name")["genes"].count().to_frame()

,genes
name,
C2.WP_THERMOGENESIS,108
C5.GOBP_ADAPTIVE_THERMOGENESIS,159
C5.GOBP_BROWN_FAT_CELL_DIFFERENTIATION,49
C5.GOBP_NEGATIVE_REGULATION_OF_COLD_INDUCED_THERMOGENESIS,47
C5.GOBP_POSITIVE_REGULATION_OF_COLD_INDUCED_THERMOGENESIS,98
REACTOME_AMPK_inhibits_chREBP_R-HSA-163680,8
REACTOME_Integration_of_energy_metabolism_R-HSA-163685,108
REACTOME_Mitochondrial_biogenesis_R-HSA-1592230,95
emont.hAd6,22


### Understanding `{...}_ThermoScores_cell.csv`

In [ ]:
def _load_thermogenic_cell_scores(dataset: Literal["TF150", "TF15_MOI2"], data_directory_path="data") -> pd.DataFrame:
    return pd.read_csv(os.path.join(data_directory_path, f"{dataset}_ThermoScores_cell.csv"))

The EWSC z-scored each signature's enrichment score across all adipocyte cells (independently per signature).

Each row is a cell, identified by `cell_id` and its assigned perturbation (under column `gene`). 

In [ ]:
tf150_cell_scores = _load_thermogenic_cell_scores("TF150")
tf150_cell_scores

,cell_id,gene,C2.WP_THERMOGENESIS,C5.GOBP_ADAPTIVE_THERMOGENESIS,C5.GOBP_BROWN_FAT_CELL_DIFFERENTIATION,C5.GOBP_NEGATIVE_REGULATION_OF_COLD_INDUCED_THERMOGENESIS,C5.GOBP_POSITIVE_REGULATION_OF_COLD_INDUCED_THERMOGENESIS,REACTOME_AMPK_inhibits_chREBP_R-HSA-163680,REACTOME_Integration_of_energy_metabolism_R-HSA-163685,REACTOME_Mitochondrial_biogenesis_R-HSA-1592230,...,z_C5.GOBP_BROWN_FAT_CELL_DIFFERENTIATION,z_C5.GOBP_NEGATIVE_REGULATION_OF_COLD_INDUCED_THERMOGENESIS,z_C5.GOBP_POSITIVE_REGULATION_OF_COLD_INDUCED_THERMOGENESIS,z_REACTOME_AMPK_inhibits_chREBP_R-HSA-163680,z_REACTOME_Integration_of_energy_metabolism_R-HSA-163685,z_REACTOME_Mitochondrial_biogenesis_R-HSA-1592230,z_emont.hAd6,z_lit.thermogenic,z_nonucp1.all,z_yi.Thermogenesis_Village
0,TRIM5_P1_AAACCATTCGCGACAT-1,TRIM5,-0.003281,-0.165548,-0.182805,0.100695,-0.355853,-1.354932,-0.039992,-0.255526,...,-0.666083,0.619552,-1.327917,-1.854393,-0.238921,-1.261375,1.973683,-1.722184,-0.935828,-0.481784
1,TRRAP_P1_AAACGAATCAATTGGT-1,TRRAP,-0.159773,-0.010466,-0.107825,0.012969,-0.099590,-0.918703,-0.259996,0.012365,...,-0.392878,0.079795,-0.371635,-1.257359,-1.553278,0.061041,0.197217,0.441466,-1.527800,-0.685218
2,RB1_P1_AAACGAATCACATGCA-1,RB1,-0.124871,-0.102575,-0.454944,0.247476,-0.241037,-0.009336,0.030142,-0.163258,...,-1.657672,1.522659,-0.899463,-0.012777,0.180075,-0.805907,0.217234,-1.217378,-0.919476,-1.272360
3,ZFP2_P1_AAACGAATCATGTGCA-1,ZFP2,0.072001,0.046005,-0.408026,-0.075729,0.003908,-0.887196,0.029654,-0.134763,...,-1.486717,-0.465942,0.014582,-1.214238,0.177157,-0.665244,-0.395776,-0.869903,-0.711210,-0.529217
4,NC_P1_AAACGAATCCCGCACT-1,NC,-0.007289,-0.047159,-0.138639,0.119276,-0.179928,-0.373886,-0.078040,-0.154032,...,-0.505157,0.733874,-0.671427,-0.511710,-0.466229,-0.760362,0.535911,-0.300625,-1.088067,-0.879569
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25291,SMAD2_P8_TGTGCCTCAACTTCGC-1,SMAD2,-0.059383,-0.104211,-0.590624,0.089276,-0.181851,-0.046220,-0.222569,0.100533,...,-2.152045,0.549296,-0.678604,-0.063258,-1.329679,0.496272,-0.725525,0.843192,0.556008,-0.340177
25292,FAM136A_P8_TGTGCGCGTTGAGCGG-1,FAM136A,-0.072470,-0.347838,-0.377287,-0.279681,-0.388578,-0.510275,0.000740,-0.057135,...,-1.374714,-1.720808,-1.450034,-0.698375,0.004423,-0.282039,-0.644783,-0.552998,-0.818189,-1.227255
25293,MBD3_P8_TGTGCTGGTCAAGGGT-1,MBD3,0.021602,0.158816,0.337255,0.114980,0.262323,0.727452,0.050917,0.165952,...,1.228848,0.707444,0.978896,0.995608,0.304192,0.819205,1.320612,0.362482,0.247852,0.233874
25294,IRF4_P8_TGTGGTTGTGGGAACT-1,IRF4,0.246234,0.127073,-0.044702,0.053277,0.222202,0.354113,0.140034,0.038076,...,-0.162878,0.327798,0.829180,0.484648,0.836597,0.187960,0.421510,-0.157827,0.375034,1.466820


The raw enrichment score for each of the 12 signatures appears as a named column (e.g., `C2.WP_THERMOGENESIS`), and the corresponding z-score computed across all adipocyte cells is provided in the `z_`-prefixed column (e.g., `z_C2.WP_THERMOGENESIS`).

In [37]:
first_row = tf150_cell_scores.iloc[0]

print("Cell ID:", first_row["cell_id"])
print("Perturbation (gene):", first_row["gene"])

pd.DataFrame([
    {
        "signature": signature,
        "raw": first_row[signature],
        "z-scored": first_row[f"z_{signature}"]
    }
    for signature in signature_names
]).set_index("signature")

Cell ID: TRIM5_P1_AAACCATTCGCGACAT-1
Perturbation (gene): TRIM5


,raw,z-scored
signature,,
nonucp1.all,-0.623876,-0.935828
REACTOME_AMPK_inhibits_chREBP_R-HSA-163680,-1.354932,-1.854393
REACTOME_Integration_of_energy_metabolism_R-HSA-163685,-0.039992,-0.238921
REACTOME_Mitochondrial_biogenesis_R-HSA-1592230,-0.255526,-1.261375
emont.hAd6,0.647249,1.973683
lit.thermogenic,-0.838709,-1.722184
C2.WP_THERMOGENESIS,-0.003281,-0.024472
C5.GOBP_ADAPTIVE_THERMOGENESIS,-0.165548,-1.000037
C5.GOBP_BROWN_FAT_CELL_DIFFERENTIATION,-0.182805,-0.666083


The same, but for TF15 MOI2:

In [36]:
tf15_moi2_cell_scores = _load_thermogenic_cell_scores("TF15_MOI2")
tf15_moi2_cell_scores

,cell_id,gene,C2.WP_THERMOGENESIS,C5.GOBP_ADAPTIVE_THERMOGENESIS,C5.GOBP_BROWN_FAT_CELL_DIFFERENTIATION,C5.GOBP_NEGATIVE_REGULATION_OF_COLD_INDUCED_THERMOGENESIS,C5.GOBP_POSITIVE_REGULATION_OF_COLD_INDUCED_THERMOGENESIS,REACTOME_AMPK_inhibits_chREBP_R-HSA-163680,REACTOME_Integration_of_energy_metabolism_R-HSA-163685,REACTOME_Mitochondrial_biogenesis_R-HSA-1592230,...,z_C5.GOBP_BROWN_FAT_CELL_DIFFERENTIATION,z_C5.GOBP_NEGATIVE_REGULATION_OF_COLD_INDUCED_THERMOGENESIS,z_C5.GOBP_POSITIVE_REGULATION_OF_COLD_INDUCED_THERMOGENESIS,z_REACTOME_AMPK_inhibits_chREBP_R-HSA-163680,z_REACTOME_Integration_of_energy_metabolism_R-HSA-163685,z_REACTOME_Mitochondrial_biogenesis_R-HSA-1592230,z_emont.hAd6,z_lit.thermogenic,z_nonucp1.all,z_yi.Thermogenesis_Village
0,P1_AAACGATGTTAGGCCC-1,NC,-0.101368,-0.056513,0.194400,0.131787,-0.084755,0.347061,-0.048267,0.203810,...,0.581161,0.534924,-0.253176,0.396434,-0.190915,0.729950,-0.076467,-0.285138,-0.876962,-0.314043
1,P1_AAACGTAAGCTGGAAG-1,NR3C1,-0.333342,-0.535980,-0.557493,-0.169546,-0.768177,-1.235179,-0.403763,0.020282,...,-1.666631,-0.688183,-2.294655,-1.410894,-1.597048,0.072641,-1.496330,-0.186200,-1.401324,-1.828192
2,P1_AAACGTCCAAGTATGC-1,KLF15,-0.032948,-0.251659,-0.571907,0.437740,-0.734765,0.659008,-0.054740,-0.166612,...,-1.709721,1.776783,-2.194848,0.752758,-0.216519,-0.596725,-0.423142,-0.883700,-1.079287,-1.426363
3,P1_AAACGTCCATCGACTC-1,KLF15,0.294086,0.399785,0.416552,0.260974,0.516443,1.598184,0.142712,0.473428,...,1.245286,1.059292,1.542689,1.825540,0.564486,1.695596,-0.376114,1.763748,1.841511,1.470416
4,P1_AAACTCACAATATCTC-1,STAT5A,-0.057976,0.015853,0.352866,0.237720,0.026088,-0.649140,-0.075851,-0.026740,...,1.054896,0.964904,0.077930,-0.741486,-0.300023,-0.095771,1.063413,-0.843928,-0.705651,-0.606054
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26423,P12_TGTGCCTCAATACAGC-1,SF3B1,-0.165442,0.153692,0.179735,-0.088389,0.356990,1.425613,0.358561,-0.050073,...,0.537320,-0.358772,1.066379,1.628419,1.418254,-0.179339,0.537786,-0.198295,0.884978,0.221658
26424,P12_TGTGGTTGTTAACGTC-1,NC,-0.057887,-0.409341,-0.411245,-0.044360,-0.637852,-0.508777,-0.005306,-0.269354,...,-1.229421,-0.180055,-1.905353,-0.581156,-0.020987,-0.964699,-1.560325,-2.277744,0.459786,-1.303683
26425,P12_TGTGTACGTCATGTCG-1,NC,-0.103364,0.130448,-0.274463,-0.331219,0.413227,1.114696,-0.232701,0.209296,...,-0.820509,-1.344412,1.234369,1.273272,-0.920429,0.749599,0.352190,1.868791,1.843207,0.051303
26426,P12_TGTGTTAGTGATGCCC-1,KLF15+PPARG,-0.195404,-0.067315,0.102394,0.018322,-0.281726,-1.462499,-0.261845,-0.259950,...,0.306108,0.074367,-0.841556,-1.670552,-1.035703,-0.931018,0.484137,0.283259,-1.072862,-0.850558


In [35]:
first_row = tf15_moi2_cell_scores.iloc[0]

print("Cell ID:", first_row["cell_id"])
print("Perturbation (gene):", first_row["gene"])

pd.DataFrame([
    {
        "signature": signature,
        "raw": first_row[signature],
        "z-scored": first_row[f"z_{signature}"]
    }
    for signature in signature_names
]).set_index("signature")

Cell ID: P1_AAACGATGTTAGGCCC-1
Perturbation (gene): NC


,raw,z-scored
signature,,
nonucp1.all,-0.645724,-0.876962
REACTOME_AMPK_inhibits_chREBP_R-HSA-163680,0.347061,0.396434
REACTOME_Integration_of_energy_metabolism_R-HSA-163685,-0.048267,-0.190915
REACTOME_Mitochondrial_biogenesis_R-HSA-1592230,0.203810,0.729950
emont.hAd6,-0.037093,-0.076467
lit.thermogenic,-0.174597,-0.285138
C2.WP_THERMOGENESIS,-0.101368,-0.532238
C5.GOBP_ADAPTIVE_THERMOGENESIS,-0.056513,-0.262066
C5.GOBP_BROWN_FAT_CELL_DIFFERENTIATION,0.194400,0.581161


### Understanding `{...}_ThermoScores_perturbation.csv`

In [38]:
def _load_thermogenic_perturbation_scores(dataset: Literal["TF150", "TF15_MOI2"], data_directory_path="data") -> pd.DataFrame:
    return pd.read_csv(os.path.join(data_directory_path, f"{dataset}_ThermoScores_perturbation.csv"))

Each row is a perturbation.

In [39]:
tf150_perturbation_scores = _load_thermogenic_perturbation_scores("TF150")
tf150_perturbation_scores

,gene,REACTOME_AMPK_inhibits_chREBP_R-HSA-163680,REACTOME_Integration_of_energy_metabolism_R-HSA-163685,REACTOME_Mitochondrial_biogenesis_R-HSA-1592230,emont.hAd6,lit.thermogenic,C2.WP_THERMOGENESIS,C5.GOBP_ADAPTIVE_THERMOGENESIS,C5.GOBP_BROWN_FAT_CELL_DIFFERENTIATION,C5.GOBP_NEGATIVE_REGULATION_OF_COLD_INDUCED_THERMOGENESIS,C5.GOBP_POSITIVE_REGULATION_OF_COLD_INDUCED_THERMOGENESIS,yi.Thermogenesis_Village,nonucp1.all,agg_top3_z
0,AATF,0.032492,-0.048839,0.016460,0.099864,0.029666,-0.096786,0.144125,0.036896,-0.075985,0.168407,0.031811,0.070473,0.137465
1,ABT1,0.205905,0.289114,0.093452,0.191029,0.004918,0.008556,0.118632,0.012386,-0.001575,0.169284,0.134745,0.150836,0.228683
2,AFF1,0.050222,-0.007968,-0.003641,0.100056,-0.044811,-0.142398,0.028565,0.055092,0.174501,-0.031410,-0.090037,-0.039105,0.109883
3,ANKRA2,0.107929,0.106782,0.118795,0.081611,0.078483,0.118346,0.075672,0.270180,-0.077010,0.110471,0.086230,0.045574,0.169107
4,BCL6,-0.049055,-0.145693,-0.085789,-0.164847,0.005943,-0.108833,-0.004936,0.101702,0.121393,-0.049118,-0.051548,-0.080945,0.076346
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123,ZNF331,0.044282,-0.042800,0.068462,0.019130,0.025743,-0.021814,-0.127693,-0.063132,-0.076692,-0.081296,-0.060660,-0.026360,0.046163
124,ZNF334,-0.179458,-0.070531,-0.155094,-0.128704,-0.224844,0.047387,-0.174215,-0.128250,-0.042564,-0.189927,-0.105869,-0.179340,-0.021903
125,ZNF354B,-0.047010,0.038747,-0.159684,-0.138273,-0.327777,-0.094722,-0.073240,-0.150210,0.083445,-0.111153,-0.070331,-0.096089,0.025061
126,ZNF419,-0.060167,-0.063237,-0.042211,0.153101,-0.058691,-0.163085,-0.103627,-0.034877,0.028448,-0.109378,-0.121416,-0.092193,0.048891


The signature columns (named according to the signature gene sets, 12 in total) contain the mean z-score across all cells belonging to that perturbation.

In [40]:
first_row = tf150_perturbation_scores.iloc[0]

print("Perturbation (gene):", first_row["gene"])
print("Aggregated top-3 z-scores:", first_row["agg_top3_z"])

pd.DataFrame([
    {
        "signature": signature,
        "z-score": first_row[signature],
    }
    for signature in signature_names
]).set_index("signature")

Perturbation (gene): AATF
Aggregated top-3 z-scores: 0.1374652489352424


,z-score
signature,
nonucp1.all,0.070473
REACTOME_AMPK_inhibits_chREBP_R-HSA-163680,0.032492
REACTOME_Integration_of_energy_metabolism_R-HSA-163685,-0.048839
REACTOME_Mitochondrial_biogenesis_R-HSA-1592230,0.016460
emont.hAd6,0.099864
lit.thermogenic,0.029666
C2.WP_THERMOGENESIS,-0.096786
C5.GOBP_ADAPTIVE_THERMOGENESIS,0.144125
C5.GOBP_BROWN_FAT_CELL_DIFFERENTIATION,0.036896


The column `agg_top3_z` is the final aggregated score -the mean of the top 3 signature z-scores- used to rank perturbations.

In [41]:
float(
    first_row

    # Select only the columns corresponding to the scores of the signatures
    [signature_names]

    # Sort the scores with the best one first
    .sort_values(ascending=False)

    # Take the top 3
    [:3]

    # Compute the mean of the top 3 scores
    .mean()
)

0.13746524892561657

The same, but for TF15 MOI2:

In [42]:
tf15_moi2_perturbation_scores = _load_thermogenic_perturbation_scores("TF15_MOI2")
tf15_moi2_perturbation_scores

,gene,REACTOME_AMPK_inhibits_chREBP_R-HSA-163680,REACTOME_Integration_of_energy_metabolism_R-HSA-163685,REACTOME_Mitochondrial_biogenesis_R-HSA-1592230,emont.hAd6,lit.thermogenic,C2.WP_THERMOGENESIS,C5.GOBP_ADAPTIVE_THERMOGENESIS,C5.GOBP_BROWN_FAT_CELL_DIFFERENTIATION,C5.GOBP_NEGATIVE_REGULATION_OF_COLD_INDUCED_THERMOGENESIS,C5.GOBP_POSITIVE_REGULATION_OF_COLD_INDUCED_THERMOGENESIS,yi.Thermogenesis_Village,nonucp1.all,agg_top3_z
0,CEBPA,-0.110602,-0.137029,-0.184693,-0.046295,-0.158806,-0.098207,-0.153996,-0.134168,0.035479,-0.181328,-0.136788,-0.156021,-0.036341
1,CEBPA+CEBPA,0.085332,-0.186797,0.085097,0.199138,0.231441,-0.402109,0.131150,-0.389938,0.356070,-0.067129,-0.258437,-0.306764,0.262216
2,CEBPA+CEBPA+STAT5B,0.099400,0.540611,0.207123,0.597711,0.855942,0.052560,0.355700,0.214564,0.854231,0.132153,0.533712,0.172513,0.769295
3,CEBPA+CEBPB,-0.248187,-0.079323,-0.326384,-0.122831,-0.275392,-0.565370,-0.605791,-0.536514,0.023794,-0.620316,-0.554110,-0.374969,-0.059453
4,CEBPA+CEBPD,0.064030,-0.031674,-0.018799,0.118689,-0.103779,-0.268750,-0.231641,0.183827,-0.205004,-0.165280,-0.157677,-0.461241,0.122182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
174,TCF7L2,-0.096740,-0.025942,-0.206074,0.071730,-0.233275,-0.071731,-0.101798,-0.123538,-0.017423,-0.100102,-0.092757,-0.117939,0.009455
175,TCF7L2+TCF7L2,0.572538,0.310538,-0.203198,0.069904,-0.379461,-0.589749,0.014998,0.363516,0.396914,0.036129,-0.447040,0.125830,0.444323
176,TCF7L2+ZBED3,-0.172731,0.007911,-0.444352,0.087612,-0.426660,0.069201,-0.138086,-0.669895,0.328217,-0.344985,0.105574,-0.157292,0.173801
177,ZBED3,-0.018980,-0.033788,-0.061123,-0.038223,-0.053244,0.019879,-0.025806,-0.012198,-0.034081,-0.007164,-0.032069,-0.060045,0.000172


In [43]:
first_row = tf15_moi2_perturbation_scores.iloc[0]

print("Perturbation (gene):", first_row["gene"])
print("Aggregated top-3 z-scores:", first_row["agg_top3_z"])

pd.DataFrame([
    {
        "signature": signature,
        "z-score": first_row[signature],
    }
    for signature in signature_names
]).set_index("signature")

Perturbation (gene): CEBPA
Aggregated top-3 z-scores: -0.036341063


,z-score
signature,
nonucp1.all,-0.156021
REACTOME_AMPK_inhibits_chREBP_R-HSA-163680,-0.110602
REACTOME_Integration_of_energy_metabolism_R-HSA-163685,-0.137029
REACTOME_Mitochondrial_biogenesis_R-HSA-1592230,-0.184693
emont.hAd6,-0.046295
lit.thermogenic,-0.158806
C2.WP_THERMOGENESIS,-0.098207
C5.GOBP_ADAPTIVE_THERMOGENESIS,-0.153996
C5.GOBP_BROWN_FAT_CELL_DIFFERENTIATION,-0.134168


In [44]:
float(first_row[signature_names].sort_values(ascending=False)[:3].mean())

-0.03634106266666667

## Your model

This notebook will only contain a dummy (random) model to give you an example of what and how to predict.

**It should not be submitted without modifications.**

### The `infer()` Function

The `infer()` function performs model inference on the dataset. <br />
The approach is different from other years, as this time there is no train function, everything must be done in infer directly.

During inference, the function:

1. Loads the perturbations to predict.
2. **Predict the mean z-scored enrichment score for each of the 12 thermogenic gene sets per perturbation.**
3. **Derive the final score by averaging the top 3 scores per perturbation.**
4. **Rank perturbations from highest to lowest.**
5. Constructs a `DataFrame` object containing the predicted gene ranking.
6. Saves the prediction in the correct structure expected by the specification.

[Expected Output](https://docs.crunchdao.com/competitions/competitions/broad-obesity/crunch-3#expected-output):

- `Report.md`: At the __end of the notebook__ 👇👇👇, write a small document that should include sufficient details outlining the approaches used by your algorithm to compute the scores.

- `prediction.parquet`: A tabular file containing predicted gene ranking for 2991 gene perturbations indicated in `predict_perturbations_3.parquet`. <br />
  Predictions for each signature should be stored in dedicated columns. <br />
  The file with predictions should have dimensions [4,474,413 × 15] (gene pairs × columns).

> **📝 Note**
> ---
> Make sure that the `Report.md` file properly documents your model, so that the Broad Institute team can reference your work in their publications.

In [15]:
def infer(
    data_directory_path: str,
    prediction_file_path: str,
    model_directory_path: str,
):
    """
    Run inference for a set of perturbations.

    Parameters:
        data_directory_path: str
            Path to the training AnnData file.
        prediction_file_path: str
            Direct path where to write the `prediction.parquet` file.
        model_directory_path: str
            Directory containing your persisted model files.

    Return:
      None: Returned value is ignored.

    Expected files:
        prediction_directory_path / "prediction.parquet": pd.DataFrame
            DataFrame (index=False) containing your predictions for each gene pairs, their score and ranking.
    """

    signatures_df = _load_thermogenic_signatures(data_directory_path)
    signatures = signatures_df["name"].tolist()

    pairs_df = _load_predict_perturbations(data_directory_path)
    # Internal note: the columns are stored as categories, so we need to convert them to strings before concatenating
    gene_pair_ids = pairs_df["gene_1"].astype(str) + "+" + pairs_df["gene_2"].astype(str)

    # Initialize random number generator with a fixed seed for reproducibility
    rng = np.random.default_rng(42)

    # Create an empty DataFrame to store predictions
    prediction = pd.DataFrame({
        "GenePairID": gene_pair_ids.values
    })

    # Keep only top-3 running max per row for FinalAggScore
    top3 = np.full((len(gene_pair_ids), 3), -np.inf)

    for sig in tqdm(signatures):
        col = rng.uniform(-2, 2, size=len(gene_pair_ids))

        # Convert to float32 to save memory
        prediction[sig] = col.astype(np.float32)

        # Update rolling top-3
        combined = np.column_stack([top3, col])
        top3 = np.sort(combined, axis=1)[:, -3:]

    # Compute FinalAggScore as the mean of the top-3 scores
    prediction["FinalAggScore"] = top3.mean(axis=1)

    # Rank gene pairs based on FinalAggScore, with rank 1 being the highest score
    prediction["Rank"] = prediction["FinalAggScore"].rank(ascending=False, method="min").astype(np.int32)

    # Write the prediction to a parquet file
    prediction.to_parquet(prediction_file_path, index=False)

## Local testing

To make sure your `infer()` function is working properly, you can call the `crunch_tools.test()` function that will reproduce the cloud environment locally. <br />
Even if it is not perfect, it should give you a quick idea if your model is working properly.

In [ ]:
crunch_tools.test()

## Results

Once the local tester is done, you can preview the result stored in `prediction/prediction.parquet`.

In [17]:
prediction = pd.read_parquet(os.path.join("prediction", "prediction.parquet"))
prediction

,GenePairID,nonucp1.all,REACTOME_AMPK_inhibits_chREBP_R-HSA-163680,REACTOME_Integration_of_energy_metabolism_R-HSA-163685,REACTOME_Mitochondrial_biogenesis_R-HSA-1592230,emont.hAd6,lit.thermogenic,C2.WP_THERMOGENESIS,C5.GOBP_ADAPTIVE_THERMOGENESIS,C5.GOBP_BROWN_FAT_CELL_DIFFERENTIATION,C5.GOBP_NEGATIVE_REGULATION_OF_COLD_INDUCED_THERMOGENESIS,C5.GOBP_POSITIVE_REGULATION_OF_COLD_INDUCED_THERMOGENESIS,yi.Thermogenesis_Village,FinalAggScore,Rank
0,AACS+AACS,0.776485,-0.743054,1.776339,-1.996716,-1.409234,0.850119,1.906193,-0.979220,0.020301,1.114637,0.454456,1.264211,1.982319,4078614
1,AACS+AATF,1.115917,-1.071427,-0.889817,1.989376,0.816667,1.327895,0.573605,1.427613,1.165559,-1.501708,-0.593401,0.940226,1.995077,1040601
2,AACS+ABCA1,-1.756300,-0.304360,-0.549982,-0.538449,1.740306,-0.096148,-0.743501,-1.618029,-1.061252,-0.753126,0.978544,-0.374288,1.993656,1589126
3,AACS+ABCB8,-1.090610,-0.543420,-1.200799,0.522910,-1.955353,-1.642248,-0.497488,0.041789,0.093482,0.561030,0.706387,-0.585605,1.994864,1122367
4,AACS+ABCB9,1.708705,-0.825252,-0.265779,-1.099211,1.677418,-1.137165,-1.056467,1.866165,-0.477400,-0.843453,1.443381,0.332977,1.975462,4376238
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4474408,ZSCAN5A+ZXDC,-0.070159,0.818683,-1.000465,-0.076704,-1.148986,-0.350737,0.190043,-1.788395,-1.153754,-0.597281,0.926985,-0.455619,1.987393,3429312
4474409,ZSCAN5A+ZZEF1,-0.855400,0.906119,1.985115,-0.116026,1.485433,-0.347313,-1.993243,-0.536228,-1.656216,-1.349133,0.269207,-0.070484,1.992042,2182547
4474410,ZXDC+ZXDC,-1.789879,1.255561,-0.837255,1.858116,-0.850971,1.632138,1.890594,-0.544307,0.610887,-0.433584,1.002509,0.599491,1.989381,2987873
4474411,ZXDC+ZZEF1,0.779938,-1.758707,1.903555,1.363800,1.446061,-1.769460,-1.305418,-0.845888,0.914151,0.523613,-1.981940,-1.334043,1.988237,3257956


### No scoring until the wet lab experiment

The aim of this competition is to help researchers identify new gene pairs, the validity of which can only be confirmed through laboratory experimentation.

## Writing the report

The final step is to write the method description as specified [in the documentation](https://docs.crunchdao.com/competitions/competitions/broad-obesity/crunch-3#file-report.md).

You must:
1. Describe the algorithm used to generate the predicted scores. *(5-10 sentences)*
2. Explain the rationale behind the proposed gene panel design. *(5-10 sentences)*
3. Describe the datasets, prior knowledge, and any additional resources used. *(5-10 sentences)*
4. An optional reference list.

The limit is about one page.
<br />
<br />
<br />

---

The report will be extracted upon submission, [see how embedded files work](https://docs.crunchdao.com/competitions/participate/notebook-processor#embed-files).

<span style="font-size: 48px">👇👇👇</span> (double-click the markdown cell below)

---
file: Report.md
---

<!-- Don't forget to change me -->

# Method Description

Describe the algorithm used to generate the predicted scores. (5-10 sentences)

# Design Rationale

Describe the reasoning behind your gene panel design. (5-10 sentences)

# Data and resources

Describing the datasets, prior knowledge, and any additional resources used. (5-10 sentences)

<span style="font-size: 48px">👆👆👆</span>

---
<br />
<br />
<br />

# Submit your Notebook

To submit your work, you must:
1. Download your Notebook from Colab
2. Upload it to the platform
3. Create a run to validate it

Executing the cell below will take care of everything (only available on Google Colab), or show you how to submit manually.

In [ ]:
# @title  {"display-mode":"form", "form-width":"400px"}

# @markdown Describe your changes, then run the cell.
Message = "" # @param {"type":"string","placeholder":"Short description (optional)"}

# ---
# THIS METHOD IS ONLY POSSIBLE ON COLAB.
# RUNNING THIS CELL WILL PROMPT YOU TO USE THE OLD WAY OF SUBMITTING A NOTEBOOK.

crunch_tools.submit(
    message=Message,
)